In [2]:
from utils.analysis.valuation import (
    CompanyAnalyzer,
    ComparisonAnalyzer,
    SectorAnalyzer,
    AnalysisWeights,
    CompanyReporter
)

In [3]:
custom_weights = AnalysisWeights(
    profitability=0.30,
    financial_health=0.30,
    growth=0.15,
    efficiency=0.10,
    valuation=0.15
)

analyzer = CompanyAnalyzer(weights=custom_weights)
comparator = ComparisonAnalyzer(company_analyzer=analyzer)
sector_analyzer = SectorAnalyzer(company_analyzer=analyzer)
reporter = CompanyReporter()

In [9]:
result = analyzer.analyze("INCY")
print(reporter.render(result))

ANÁLISIS: Incyte Corporation (INCY)
Sector: Healthcare | Industria: Biotechnology
País: United States | Moneda: USD

─────────────────────────────────────────────────────────────────
RESUMEN DE SCORES
─────────────────────────────────────────────────────────────────
  🟢 Rentabilidad       [███████████████████░]   97.7
  🟢 Salud Financiera   [███████████████████░]   97.1
  🟢 Crecimiento        [████████████████░░░░]   83.3
  🟠 Eficiencia         [████████░░░░░░░░░░░░]   41.4
  🟠 Valoración         [███████████░░░░░░░░░]   59.2
─────────────────────────────────────────────────────────────────
  🟢 TOTAL              [████████████████░░░░]   84.0
  📋 Conclusión: EXCELENTE

─────────────────────────────────────────────────────────────────
RENTABILIDAD (Score: 97.7)
─────────────────────────────────────────────────────────────────
  ROIC:             45.1%        excellent
  ROE:              30.4%        excellent
  ROA:              13.5%        good
  Margen Bruto:     56.5%        excell

In [5]:
TICKERS = ["TSM", "PYPL"]

comparison = comparator.compare(TICKERS)
display(comparison['summary_df'])

,Ticker,Nombre,Sector,Rentabilidad,Salud Fin.,Crecimiento,Eficiencia,Valoración,TOTAL,Conclusión
0,TSM,Taiwan Semiconductor Manu,Technology,99.847556,86.765959,85.104167,45.644697,44.934135,80.054269,EXCELENTE
1,PYPL,"PayPal Holdings, Inc.",Financial Servi,74.351560,64.333844,61.062500,6.433851,79.513878,63.335463,REGULAR


In [6]:
for item in comparison['ranking']:
    emoji = "🥇" if item['rank'] == 1 else "🥈" if item['rank'] == 2 else "🥉" if item['rank'] == 3 else "  "
    print(f"{emoji} #{item['rank']} {item['ticker']:<6} {item['name'][:20]:<20} Score: {item['score']:.1f} → {item['conclusion']}")

🥇 #1 TSM    Taiwan Semiconductor Score: 80.1 → EXCELENTE
🥈 #2 PYPL   PayPal Holdings, Inc Score: 63.3 → REGULAR


In [7]:
category_names = {
    'profitability': 'Rentabilidad',
    'financial_health': 'Salud Financiera',
    'growth': 'Crecimiento',
    'efficiency': 'Eficiencia',
    'valuation': 'Valoración'
}

for cat, data in comparison['category_leaders'].items():
    name = category_names.get(cat, cat)
    best = data['best']
    worst = data['worst']
    print(f"\n{name}:")
    print(f"  🟢 Mejor:  {best['ticker']:<6} ({best['score']:.1f})")
    print(f"  🔴 Peor:   {worst['ticker']:<6} ({worst['score']:.1f})")


Rentabilidad:
  🟢 Mejor:  TSM    (99.8)
  🔴 Peor:   PYPL   (74.4)

Salud Financiera:
  🟢 Mejor:  TSM    (86.8)
  🔴 Peor:   PYPL   (64.3)

Crecimiento:
  🟢 Mejor:  TSM    (85.1)
  🔴 Peor:   PYPL   (61.1)

Eficiencia:
  🟢 Mejor:  TSM    (45.6)
  🔴 Peor:   PYPL   (6.4)

Valoración:
  🟢 Mejor:  PYPL   (79.5)
  🔴 Peor:   TSM    (44.9)


In [8]:
print("POSICIÓN RELATIVA VS PEERS")

TECH_PEERS = ["AAPL", "MSFT", "GOOGL", "AMZN"]

sector_result = sector_analyzer.analyze_vs_peers(
    ticker="META",
    peers=TECH_PEERS
)

rel_pos = sector_result['relative_position']

for cat, data in rel_pos.items():
    if isinstance(data, dict) and 'company_score' in data:
        name = category_names.get(cat, cat.upper())
        diff = data['vs_avg']
        arrow = "↑" if diff > 0 else "↓" if diff < 0 else "="
        color = "🟢" if diff > 5 else "🔴" if diff < -5 else "🟡"
        
        print(f"\n{name}:")
        print(f"  Tu score:    {data['company_score']:.1f}")
        print(f"  Promedio:    {data['peer_avg']:.1f}")
        print(f"  {color} Diferencia: {diff:+.1f} {arrow}")
        print(f"  Posición:    #{data['rank']} de {data['total_compared']}")

POSICIÓN RELATIVA VS PEERS

Rentabilidad:
  Tu score:    98.8
  Promedio:    90.0
  🟢 Diferencia: +8.8 ↑
  Posición:    #3 de 5

Salud Financiera:
  Tu score:    77.7
  Promedio:    68.5
  🟢 Diferencia: +9.2 ↑
  Posición:    #4 de 5

Crecimiento:
  Tu score:    38.5
  Promedio:    68.0
  🔴 Diferencia: -29.5 ↓
  Posición:    #1 de 5

Eficiencia:
  Tu score:    50.5
  Promedio:    54.0
  🟡 Diferencia: -3.5 ↓
  Posición:    #3 de 5

Valoración:
  Tu score:    30.2
  Promedio:    17.6
  🟢 Diferencia: +12.6 ↑
  Posición:    #5 de 5

TOTAL:
  Tu score:    68.3
  Promedio:    65.8
  🟡 Diferencia: +2.5 ↑
  Posición:    #4 de 5
